In [ ]:
# stdlib pathlib (filesystem paths)
from pathlib import Path

# src/utils/pathing.py
from src.utils.pathing import ensure_repo_root_on_sys_path  # src/utils/pathing.py

ensure_repo_root_on_sys_path(Path.cwd())

# 13. Supervised Learning: Linear Regression

## Algorithm Category
**Type**: Supervised Learning - Regression  
**Complexity**: Low  
**Use Case**: Predicting continuous numerical values

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the mathematical foundation of linear regression
- Implement linear regression using scikit-learn
- Validate model assumptions and check residuals
- Evaluate regression performance using multiple metrics
- Perform hyperparameter tuning and cross-validation
- Interpret model coefficients and feature importance
- Apply linear regression to real-world datasets

## Historical Context

Linear regression is one of the oldest and most fundamental algorithms in statistics and machine learning. It was first developed by Francis Galton in the 1880s and later formalized by Karl Pearson and others. The method of least squares, which linear regression uses, was independently developed by Carl Friedrich Gauss and Adrien-Marie Legendre in the early 19th century.

**Key Papers/References:**
- Gauss, C.F. (1809). "Theoria motus corporum coelestium"
- Legendre, A.M. (1805). "Nouvelles méthodes pour la détermination des orbites des comètes"

## When to Use Linear Regression

Linear regression is appropriate when:
- The target variable is continuous (not categorical)
- There is a linear relationship between features and target
- Features are independent (or multicollinearity is addressed)
- The dataset is not too large (computational complexity is O(n²))
- Interpretability is important (coefficients are easy to understand)


## Theory & Mechanics

### Mathematical Foundation

Linear regression models the relationship between a dependent variable (target) y and one or more independent variables (features) X using a linear function:

**Simple Linear Regression (one feature):**
$$y = \beta_0 + \beta_1 x + \epsilon$$

**Multiple Linear Regression (multiple features):**
$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_n x_n + \epsilon$$

Where:
- $y$ is the target variable
- $\beta_0$ is the intercept (bias term)
- $\beta_1, \beta_2, ..., \beta_n$ are the coefficients (weights) for each feature
- $x_1, x_2, ..., x_n$ are the feature values
- $\epsilon$ is the error term (residuals)

### How It Works

1. **Objective**: Find the coefficients that minimize the sum of squared residuals (Ordinary Least Squares - OLS)
2. **Cost Function**: Mean Squared Error (MSE)
   $$MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$
3. **Solution**: Closed-form solution using matrix algebra:
   $$\beta = (X^T X)^{-1} X^T y$$
4. **Prediction**: Once coefficients are learned, predictions are made by:
   $$\hat{y} = X\beta$$

### Key Assumptions

Linear regression assumes:
1. **Linearity**: Relationship between features and target is linear
2. **Independence**: Observations are independent of each other
3. **Homoscedasticity**: Constant variance of residuals across all feature values
4. **Normality**: Residuals are normally distributed
5. **No multicollinearity**: Features are not highly correlated with each other

### Limitations

- Cannot model non-linear relationships
- Sensitive to outliers
- Assumes linearity (may not capture complex patterns)
- Requires feature scaling for meaningful coefficient interpretation


## Implementation

Let's implement linear regression step by step using scikit-learn and our helper functions.

**Implementation Steps:**
1. **Import Libraries**: Load necessary tools
2. **Load Data**: Get dataset for training
3. **Preprocess**: Scale features, split data
4. **Train Model**: Fit linear regression to training data
5. **Make Predictions**: Use model to predict on test data
6. **Evaluate**: Calculate performance metrics
7. **Validate**: Check assumptions and model quality
8. **Interpret**: Understand what the model learned


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Setting Up Our Tools
# ============================================

# Core data science libraries
import numpy as np  # NumPy: Numerical computing (arrays, math operations)
import pandas as pd  # Pandas: Data manipulation (DataFrames, data analysis)
import matplotlib.pyplot as plt  # Matplotlib: Plotting and visualization

# Scikit-learn: Machine learning library
from sklearn.datasets import load_diabetes, make_regression  # Dataset loaders
from sklearn.linear_model import LinearRegression  # Linear regression algorithm
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV  # Model selection tools
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error  # Evaluation metrics

# ============================================
# IMPORTING OUR HELPER FUNCTIONS
# ============================================

# Our custom utility functions (organized in src/ directory)
from src.models.supervised import split_data, evaluate_regressor, cross_validate_model  # Supervised learning utilities
from src.models.regression import (
    calculate_residuals,  # Calculate prediction errors
    plot_residuals,  # Visualize residuals
    check_residual_normality,  # Test if residuals are normally distributed
    calculate_goodness_of_fit,  # Calculate R², RMSE, MAE, etc.
    check_linearity_assumptions  # Check regression assumptions
)
from src.processing.preprocessing import scale_features  # Normalize features
from src.utils.benchmarking import benchmark_model_training  # Measure training time
from src.utils.traceability import extract_feature_importance_trace, save_traceability_data  # Model interpretability
from src.utils.validation import validate_model_output, check_cross_validation_stability  # Model validation

print("Libraries imported successfully!")  # Confirm all imports worked


In [ ]:
# ============================================
# LOADING THE DATASET: Diabetes Regression Data
# ============================================

# load_diabetes() loads the Diabetes dataset from scikit-learn
# This dataset predicts disease progression (continuous value) from medical measurements
# No download needed - it's built into scikit-learn
diabetes = load_diabetes()  # Returns a Bunch object with data, target, feature_names

# X = Features (inputs): Medical measurements
# diabetes.data contains feature values (442 samples × 10 features)
# We convert to DataFrame for easier manipulation
# columns=diabetes.feature_names gives meaningful column names (age, sex, bmi, etc.)
X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
# Features include: age, sex, body mass index (bmi), blood pressure, etc.

# y = Target (output): Disease progression (what we want to predict)
# diabetes.target contains the target values (continuous numbers)
# We convert to Series and give it a descriptive name
y = pd.Series(diabetes.target, name='Disease Progression')
# Higher values = more disease progression

# ============================================
# EXPLORING THE DATASET: Understanding Our Data
# ============================================

# .shape returns (rows, columns) - dimensions of the dataset
print(f"Dataset Shape: {X.shape}")  # Output: (442, 10) - 442 patients, 10 features

# Get target name (what we're predicting)
# hasattr() checks if object has an attribute (safety check)
print(f"Target Name: {diabetes.target_names[0] if hasattr(diabetes, 'target_names') else 'Disease Progression'}")

# Display first few rows to see what the data looks like
print(f"\nFirst few rows:")
print(X.head())  # Shows first 5 rows with all feature values

# .describe() provides statistical summary of the target variable
# Shows: count, mean, std, min, max, quartiles
print(f"\nTarget statistics:")
print(y.describe())  # Statistical summary of disease progression values


In [ ]:
# ============================================
# FEATURE SCALING: Normalizing Features
# ============================================

# Why scale features?
# - Linear regression coefficients are easier to interpret when features are on same scale
# - Features with larger values would dominate otherwise
# - Scaling doesn't change relationships, just makes them comparable

# scale_features() normalizes features to have mean=0 and std=1
# fit=True means "learn scaling from this data" (use for training data)
# Returns: scaled features and the scaler (save scaler to scale test data later)
X_scaled, scaler = scale_features(X, fit=True)
# X_scaled: Features normalized (mean=0, std=1 for each column)
# scaler: The scaling object (needed to scale test data with same transformation)

# ============================================
# TRAIN/TEST SPLIT: Separating Data
# ============================================

# CRITICAL: Split AFTER scaling to prevent data leakage
# If we split first, test data statistics could influence scaling

# split_data() randomly splits data into training (80%) and test (20%) sets
# test_size=0.2 means 20% for testing, 80% for training
# random_state=42 ensures same split every time (reproducibility)
X_train, X_test, y_train, y_test = split_data(X_scaled, y, test_size=0.2, random_state=42)
# X_train: 80% of samples for training (scaled features)
# X_test: 20% of samples for testing (will be scaled using same scaler later)
# y_train: Target values for training samples
# y_test: Target values for test samples (ground truth)

# Display split information
print(f"Training set: {X_train.shape[0]} samples")  # ~354 samples (80% of 442)
print(f"Test set: {X_test.shape[0]} samples")  # ~88 samples (20% of 442)


In [ ]:
# ============================================
# MODEL CREATION: Linear Regression
# ============================================

# Create a LinearRegression model object
# This model will learn a linear relationship: y = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ
# No hyperparameters needed for basic linear regression
model = LinearRegression()

# ============================================
# MODEL TRAINING: Learning from Data
# ============================================

# .fit() trains the model on training data
# The model learns the coefficients (β values) that minimize prediction error
# Internally, it solves: β = (X^T X)^(-1) X^T y (matrix algebra)
model.fit(X_train, y_train)  # Train the model

print("Model trained successfully!")  # Confirm training completed

# ============================================
# MODEL INTERPRETATION: Understanding Coefficients
# ============================================

# model.intercept_ is β₀ (the y-intercept)
# This is the predicted value when all features are 0
print(f"Intercept: {model.intercept_:.3f}")  # Display intercept value

# model.coef_ contains β₁, β₂, ..., βₙ (coefficients for each feature)
# Each coefficient tells us: "How much does y change when this feature increases by 1?"
# Positive coefficient: feature increases → target increases
# Negative coefficient: feature increases → target decreases

print(f"\nCoefficients:")
# Create DataFrame to display coefficients nicely
coef_df = pd.DataFrame({
    'Feature': X.columns,  # Feature names (age, sex, bmi, etc.)
    'Coefficient': model.coef_  # Coefficient values (β values)
})

# Sort by absolute value of coefficient (largest impact first)
# key=abs means "sort by absolute value" (ignore sign)
# ascending=False means largest first
coef_df = coef_df.sort_values('Coefficient', key=abs, ascending=False)
print(coef_df)  # Display coefficients sorted by importance


In [ ]:
# ============================================
# MAKING PREDICTIONS: Using the Trained Model
# ============================================

# .predict() uses the trained model to make predictions
# Formula: ŷ = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ

# Predict on training data (data the model saw during training)
y_pred_train = model.predict(X_train)  # Predictions for training samples

# Predict on test data (unseen data - simulates real-world performance)
y_pred_test = model.predict(X_test)  # Predictions for test samples

# ============================================
# MODEL EVALUATION: Measuring Performance
# ============================================

# evaluate_regressor() calculates multiple regression metrics
# Returns dictionary with MSE, RMSE, and other metrics

# Evaluate on training data
train_results = evaluate_regressor(model, X_train, y_train)
# This shows how well model fits the training data

# Evaluate on test data (more important - shows real-world performance)
test_results = evaluate_regressor(model, X_test, y_test)
# This shows how well model generalizes to new data

# ============================================
# DISPLAYING PERFORMANCE METRICS
# ============================================

print("Training Performance:")
# RMSE (Root Mean Squared Error): Average prediction error in same units as target
# Lower is better (0 = perfect predictions)
print(f"  RMSE: {train_results['rmse']:.3f}")  # Training RMSE

# MSE (Mean Squared Error): Average squared prediction error
# Squared errors penalize large errors more
print(f"  MSE: {train_results['mse']:.3f}")  # Training MSE

print("\nTest Performance:")
# Test metrics are more important - they show real-world performance
print(f"  RMSE: {test_results['rmse']:.3f}")  # Test RMSE
print(f"  MSE: {test_results['mse']:.3f}")  # Test MSE

# ============================================
# R² SCORE: Coefficient of Determination
# ============================================

# R² (R-squared) measures how well model explains variance in target
# Range: -∞ to 1.0
# 1.0 = perfect predictions (model explains 100% of variance)
# 0.0 = model is no better than predicting the mean
# Negative = model is worse than predicting the mean

# Calculate R² for both training and test sets
r2_train = r2_score(y_train, y_pred_train)  # R² on training data
r2_test = r2_score(y_test, y_pred_test)  # R² on test data

print(f"\nR² Score - Train: {r2_train:.3f}, Test: {r2_test:.3f}")
# Compare train vs test R²:
# - If train R² >> test R²: Model is overfitting (memorizing training data)
# - If train R² ≈ test R²: Model generalizes well


## Validation & Testing

Let's validate our model using multiple approaches: in-notebook assertions, cross-validation, and assumption checking.

**Why Validate?**
- **Catch Errors**: Ensure model is working correctly
- **Check Assumptions**: Verify regression assumptions are met
- **Assess Stability**: Ensure model performance is consistent
- **Build Confidence**: Multiple validation methods increase trust in results

**Validation Methods:**
1. **Output Validation**: Check predictions are reasonable
2. **Cross-Validation**: Test on multiple train/test splits
3. **Assumption Checking**: Verify statistical assumptions
4. **Residual Analysis**: Check if errors are random


In [ ]:
# ============================================
# VALIDATION 1: Model Output Validity
# ============================================

# validate_model_output() checks if predictions make sense
# It verifies:
# - Predictions and true values have same shape
# - No NaN or Inf values in predictions
# - Predictions are reasonable (not all zeros, not constant)

# task_type='regression' tells validator this is a regression problem
validation_result = validate_model_output(y_pred_test, y_test.values, task_type='regression')
# Returns dictionary with validation results

print("Model Output Validation:")
print(f"  Valid: {validation_result['valid']}")  # True if predictions are valid

# If validation passed, display metrics
if 'mse' in validation_result:
    print(f"  MSE: {validation_result['mse']:.3f}")  # Mean Squared Error
    print(f"  RMSE: {validation_result['rmse']:.3f}")  # Root Mean Squared Error

# ============================================
# ASSERTIONS: Automated Checks
# ============================================

# assert statements automatically check conditions
# If condition is False, program stops with error message
# This catches bugs early and documents expected behavior

# Check 1: Predictions must be valid
assert validation_result['valid'], "Model predictions are invalid!"
# If predictions are invalid, stop execution with error message

# Check 2: RMSE must be positive (errors exist)
assert test_results['rmse'] > 0, "RMSE should be positive"
# RMSE = 0 would mean perfect predictions (unlikely in real data)

# Check 3: R² must be non-negative (for this dataset)
assert r2_test >= 0, "R² score should be non-negative"
# Negative R² means model is worse than predicting the mean

print("\n✓ Basic validation checks passed")  # All assertions passed!


In [ ]:
# ============================================
# VALIDATION 2: Cross-Validation
# ============================================

# Cross-validation splits data into k folds (groups)
# Trains on k-1 folds, tests on 1 fold
# Repeats k times (each fold used as test set once)
# More reliable than single train/test split

# cross_val_score() performs k-fold cross-validation
# cv=5 means 5 folds (5 train/test splits)
# scoring='neg_mean_squared_error' means use negative MSE as score
# (scikit-learn uses "neg" because higher scores are better, but MSE lower is better)
cv_scores = cross_val_score(model, X_scaled, y, cv=5, scoring='neg_mean_squared_error')
# Returns array of 5 scores (one per fold)

# Convert negative MSE to RMSE
# -cv_scores converts back to positive MSE
# np.sqrt() takes square root to get RMSE
cv_rmse = np.sqrt(-cv_scores)  # Array of 5 RMSE values (one per fold)

# Calculate statistics across folds
cv_mean = cv_rmse.mean()  # Average RMSE across all folds
cv_std = cv_rmse.std()  # Standard deviation (measure of variability)

print("Cross-Validation Results (5-fold):")
print(f"  Mean RMSE: {cv_mean:.3f} (+/- {cv_std:.3f})")  # Average ± variability
print(f"  Individual fold RMSEs: {cv_rmse}")  # RMSE for each of the 5 folds

# ============================================
# STABILITY CHECK: Is Performance Consistent?
# ============================================

# Check if cross-validation results are stable (low variation)
# Unstable results suggest model is sensitive to data split
# Stable results suggest model is robust

# check_cross_validation_stability() calculates coefficient of variation
# threshold=0.2 means "variation should be less than 20% of mean"
stability = check_cross_validation_stability(cv_rmse, threshold=0.2)
# Returns dictionary with stability analysis

print(f"\nCV Stability Check:")
print(f"  Coefficient of Variation: {stability['cv_coefficient']:.3f}")  # Relative variability
print(f"  Is Stable: {stability['is_stable']}")  # True if variation < threshold

# ============================================
# ASSERTIONS: Cross-Validation Checks
# ============================================

# Check 1: CV RMSE must be positive
assert cv_mean > 0, "CV RMSE should be positive"

# Check 2: Results must be stable
assert stability['is_stable'], "Cross-validation results are unstable!"
# Unstable results suggest model is unreliable

print("\n✓ Cross-validation checks passed")  # All checks passed!


In [ ]:
# ============================================
# VALIDATION 3: Checking Regression Assumptions
# ============================================

# Linear regression makes several assumptions
# If assumptions are violated, results may be unreliable
# We check these assumptions to ensure model validity

print("Checking Regression Assumptions:")
print("(Assumption checking code will follow in next cells)")
assumptions = check_linearity_assumptions(X_test.values, y_test.values, y_pred_test)
print(f"\n1. Residual-Prediction Correlation: {assumptions['residual_prediction_correlation']:.3f}")
print(f"   Homoscedasticity OK: {assumptions['homoscedasticity_ok']}")

print(f"\n2. Durbin-Watson Statistic: {assumptions['durbin_watson']:.3f}")
print(f"   Independence OK: {assumptions['independence_ok']}")

print(f"\n3. Normality Tests:")
normality = assumptions['normality']
print(f"   Shapiro-Wilk p-value: {normality['shapiro_wilk']['p_value']:.3f}")
print(f"   Is Normal (Shapiro): {normality['shapiro_wilk']['is_normal']}")
print(f"   D'Agostino p-value: {normality['dagostino']['p_value']:.3f}")
print(f"   Is Normal (D'Agostino): {normality['dagostino']['is_normal']}")

# Note: Assumptions may not always be perfectly met in practice
print("\nNote: Some assumptions may not be perfectly met, but model can still be useful.")


In [ ]:
# ============================================
# VALIDATION 4: Goodness of Fit Metrics
# ============================================

# calculate_goodness_of_fit() computes multiple regression metrics
# These metrics measure how well the model fits the data
# Returns dictionary with R², MSE, RMSE, MAE, MAPE, etc.

goodness_of_fit = calculate_goodness_of_fit(y_test.values, y_pred_test)
# Calculates all metrics comparing true values (y_test) vs predictions (y_pred_test)

print("Goodness of Fit Metrics:")
# Loop through all metrics and display them
for metric, value in goodness_of_fit.items():
    # metric = metric name (e.g., 'r2_score', 'rmse', 'mae')
    # value = metric value (the actual number)
    print(f"  {metric}: {value:.3f}")  # Display each metric with 3 decimal places

# ============================================
# ASSERTIONS: Goodness of Fit Checks
# ============================================

# Check 1: R² must be non-negative (for this dataset)
assert goodness_of_fit['r2_score'] >= 0, "R² should be non-negative"
# Negative R² means model is worse than predicting the mean

# Check 2: RMSE must be positive (errors exist)
assert goodness_of_fit['rmse'] > 0, "RMSE should be positive"
# RMSE = 0 would mean perfect predictions (unlikely)

print("\n✓ Goodness of fit checks passed")  # All checks passed!


## Performance Benchmarking

Let's benchmark the model's performance in terms of training time, prediction speed, and compare with baseline.

**Why Benchmark?**
- **Training Time**: How long does it take to train? (important for large datasets)
- **Prediction Speed**: How fast can we make predictions? (important for real-time applications)
- **Baseline Comparison**: Is our model better than a simple baseline? (must outperform mean/median predictor)

**Baseline Models:**
- **Mean Predictor**: Always predicts the average of training data
- **Median Predictor**: Always predicts the median of training data
- **Random Predictor**: Random predictions (worst case)

**Performance Metrics:**
- Training time (seconds)
- Prediction time (seconds)
- Predictions per second (throughput)
- Model accuracy (RMSE, R²)


In [ ]:
# ============================================
# PERFORMANCE BENCHMARKING: Measuring Speed
# ============================================

# benchmark_model_training() measures:
# - How long training takes
# - How long predictions take
# - How many predictions per second (throughput)
# - Model accuracy (RMSE)

# Parameters:
# - LinearRegression(): The model to benchmark (not yet trained)
# - X_train.values, y_train.values: Training data (NumPy arrays)
# - X_test.values, y_test.values: Test data (NumPy arrays)

benchmark_results = benchmark_model_training(
    LinearRegression(),  # Model class (will create new instance)
    X_train.values,  # Training features (convert DataFrame to NumPy array)
    y_train.values,  # Training targets (convert Series to NumPy array)
    X_test.values,  # Test features
    y_test.values  # Test targets
)
# Returns dictionary with timing and performance metrics

print("Performance Benchmark:")
# Display training time (how long it took to fit the model)
print(f"  Training Time: {benchmark_results['training_time']:.4f} seconds")

# Display prediction time (how long it took to predict on test set)
print(f"  Prediction Time: {benchmark_results['prediction_time']:.4f} seconds")

# Display throughput (how many predictions per second)
# Higher is better (faster predictions)
print(f"  Predictions per Second: {benchmark_results['predictions_per_second']:.0f}")

# Display model accuracy
print(f"  Test RMSE: {benchmark_results['test_rmse']:.3f}")

# Display dataset size (context for performance)
print(f"  Dataset Size: {benchmark_results['n_samples']} samples, {benchmark_results['n_features']} features")


In [ ]:
# ============================================
# BASELINE COMPARISON: Is Our Model Better?
# ============================================

# Baseline model: Always predicts the mean of training data
# This is the simplest possible model (no learning, just average)
# Our model MUST be better than this, or it's useless!

# np.full_like() creates array with same shape as y_test, filled with a constant value
# y_train.mean() calculates the average of training targets
# Result: Array of predictions, all equal to the mean
baseline_pred = np.full_like(y_test.values, y_train.mean())
# Example: If mean is 150, baseline_pred = [150, 150, 150, ..., 150]

# Calculate RMSE for baseline (how wrong is "always predict the mean"?)
baseline_rmse = np.sqrt(mean_squared_error(y_test.values, baseline_pred))
# This measures error when we just predict the average

print("Baseline Comparison:")
print(f"  Baseline (Mean) RMSE: {baseline_rmse:.3f}")  # Error of simple baseline
print(f"  Linear Regression RMSE: {test_results['rmse']:.3f}")  # Error of our model

# Calculate improvement percentage
# Formula: (baseline_error - model_error) / baseline_error * 100
# Positive = improvement, negative = worse than baseline
improvement = ((baseline_rmse - test_results['rmse']) / baseline_rmse * 100)
print(f"  Improvement: {improvement:.1f}%")  # How much better is our model?

# ============================================
# ASSERTION: Model Must Beat Baseline
# ============================================

# CRITICAL CHECK: Our model must be better than baseline
# If not, we should just use the mean predictor (simpler and faster)
assert test_results['rmse'] < baseline_rmse, "Model should outperform baseline!"
# If model is worse than baseline, stop execution with error

print("\n✓ Model outperforms baseline")  # Model is better than simple baseline!


## Traceability

Let's extract and visualize feature importance, model coefficients, and create traceability records.

**What is Traceability?**
Traceability means keeping records of:
- **What model was trained**: Algorithm, hyperparameters, dataset
- **What features were used**: Feature names, importance, coefficients
- **How well it performed**: Metrics, validation results
- **When it was trained**: Timestamp, version

**Why is it Important?**
- **Reproducibility**: Can recreate the same model later
- **Debugging**: Understand why model made certain predictions
- **Compliance**: Required for regulated industries (healthcare, finance)
- **Documentation**: Record of what was tried and what worked


In [ ]:
# ============================================
# FEATURE IMPORTANCE: Which Features Matter Most?
# ============================================

# In linear regression, feature importance = absolute value of coefficients
# Larger coefficient (positive or negative) = more impact on prediction
# extract_feature_importance_trace() extracts and organizes this information

feature_importance = extract_feature_importance_trace(
    model,  # The trained model (has .coef_ attribute)
    feature_names=X.columns.tolist()  # List of feature names (age, sex, bmi, etc.)
)
# Returns DataFrame with features sorted by importance (absolute coefficient value)

print("Feature Importance (by absolute coefficient):")
print(feature_importance)  # Display feature importance table

# ============================================
# VISUALIZING FEATURE IMPORTANCE
# ============================================

# Create horizontal bar chart showing feature importance
import matplotlib.pyplot as plt

# Create figure with specified size (width=10 inches, height=6 inches)
plt.figure(figsize=(10, 6))

# Create horizontal bar chart
# range(len(feature_importance)): Y-axis positions (0, 1, 2, ...)
# feature_importance['importance']: Bar lengths (absolute coefficient values)
# align='center': Center bars on Y-axis positions
plt.barh(range(len(feature_importance)), feature_importance['importance'], align='center')

# Set Y-axis labels to feature names
plt.yticks(range(len(feature_importance)), feature_importance['feature'])

# Label axes
plt.xlabel('Absolute Coefficient Value')  # X-axis: importance magnitude
plt.title('Feature Importance (Linear Regression Coefficients)')  # Chart title

# Invert Y-axis so most important feature is at top
plt.gca().invert_yaxis()  # gca() = get current axes

# Adjust layout to prevent label overlap
plt.tight_layout()
plt.show()  # Display the plot

# Interpretation:
# - Longer bars = more important features
# - Features at top have largest impact on predictions


In [ ]:
# ============================================
# CREATING TRACEABILITY RECORD: Documenting Everything
# ============================================

# Create a dictionary containing all important information about the model
# This record can be saved and used later to reproduce or understand the model

trace_data = {
    # Model identification
    "model_type": "LinearRegression",  # What algorithm was used
    "dataset": "Diabetes",  # What dataset was used
    
    # Dataset information
    "n_samples": len(X),  # Number of training samples (442)
    "n_features": X.shape[1],  # Number of features (10)
    
    # Model parameters (what the model learned)
    # dict(zip()) pairs feature names with their coefficients
    # Example: {'age': 0.5, 'bmi': -0.3, ...}
    "coefficients": dict(zip(X.columns, model.coef_)),
    "intercept": float(model.intercept_),  # Y-intercept (β₀)
    
    # Feature importance (sorted by importance)
    # .to_dict('records') converts DataFrame to list of dictionaries
    "feature_importance": feature_importance.to_dict('records'),
    
    # Performance metrics (how well the model performed)
    "performance_metrics": {
        "train_rmse": float(train_results['rmse']),  # Training error
        "test_rmse": float(test_results['rmse']),  # Test error (more important)
        "r2_train": float(r2_train),  # Training R²
        "r2_test": float(r2_test)  # Test R²
    },
    
    # Cross-validation results (robust performance estimate)
    "cross_validation": {
        "mean_rmse": float(cv_mean),  # Average RMSE across folds
        "std_rmse": float(cv_std)  # Standard deviation (variability)
    }
}

# ============================================
# SAVING TRACEABILITY DATA: Persisting Records
# ============================================

# save_traceability_data() saves the record to a JSON file
# File is timestamped to prevent overwriting previous records
# "linear_regression_trace" is the base filename

trace_path = save_traceability_data(trace_data, "linear_regression_trace")
# Returns path to saved file (e.g., "outputs/traceability/linear_regression_trace_20251230_143022.json")

print(f"Traceability data saved to: {trace_path}")  # Show where file was saved

# This file can be loaded later to:
# - Reproduce the same model
# - Understand what features were important
# - Compare with other models
# - Document experiments for reports


## Visualization & Diagnostics

Let's visualize the model's performance and check residuals.

**Why Visualize?**
- **See Patterns**: Visual inspection reveals issues numbers miss
- **Check Assumptions**: Residual plots show if assumptions are violated
- **Understand Errors**: See where model makes mistakes
- **Communicate Results**: Visuals are easier to understand than numbers

**Key Visualizations:**
1. **Predicted vs Actual**: How close are predictions to truth?
2. **Residual Plot**: Are errors random or systematic?
3. **Q-Q Plot**: Are residuals normally distributed?


In [ ]:
# ============================================
# VISUALIZATION 1: Predicted vs Actual
# ============================================

# Create figure with 2 subplots side by side
# figsize=(12, 5) means width=12 inches, height=5 inches
plt.figure(figsize=(12, 5))

# Subplot 1: Predicted vs Actual scatter plot
plt.subplot(1, 2, 1)  # 1 row, 2 columns, position 1 (left)

# Scatter plot: each point is (actual_value, predicted_value)
# alpha=0.6 makes points semi-transparent (easier to see overlapping points)
plt.scatter(y_test, y_pred_test, alpha=0.6)

# Perfect prediction line: y = x (diagonal line)
# If predictions were perfect, all points would lie on this line
# [y_test.min(), y_test.max()] = x-coordinates (min to max actual values)
# [y_test.min(), y_test.max()] = y-coordinates (same values for perfect line)
# 'r--' = red dashed line, lw=2 = line width 2
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)

# Label axes and add title
plt.xlabel('Actual Values')  # X-axis: true target values
plt.ylabel('Predicted Values')  # Y-axis: model predictions
plt.title('Predicted vs Actual Values')  # Chart title
plt.grid(True, alpha=0.3)  # Add grid (alpha=0.3 = semi-transparent)

# Interpretation:
# - Points close to red line = good predictions
# - Points far from red line = large errors
# - Systematic patterns = model issues (e.g., always overestimating)

# ============================================
# VISUALIZATION 2: Residual Plot
# ============================================

# Residuals = actual - predicted (prediction errors)
# calculate_residuals() computes: residuals = y_true - y_pred
residuals = calculate_residuals(y_test.values, y_pred_test)

# Subplot 2: Residual plot
plt.subplot(1, 2, 2)  # 1 row, 2 columns, position 2 (right)

# Scatter plot: residuals vs predicted values
# X-axis: predicted values, Y-axis: residuals (errors)
plt.scatter(y_pred_test, residuals, alpha=0.6)

# Horizontal line at y=0 (no error)
# Residuals should be randomly scattered around this line
plt.axhline(y=0, color='r', linestyle='--')  # Red dashed line at y=0

# Label axes and add title
plt.xlabel('Predicted Values')  # X-axis: model predictions
plt.ylabel('Residuals')  # Y-axis: prediction errors
plt.title('Residual Plot')  # Chart title
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout to prevent label overlap
plt.tight_layout()
plt.show()  # Display both plots

# Interpretation of residual plot:
# - Random scatter around y=0 = good (errors are random)
# - Funnel shape = heteroscedasticity (variance changes with predictions)
# - Curved pattern = non-linearity (model missing non-linear relationships)
# - Systematic pattern = model bias (e.g., always underestimating high values)


In [ ]:
# ============================================
# VISUALIZATION 3: Residual Analysis with Q-Q Plot
# ============================================

# plot_residuals() creates comprehensive residual analysis plots
# Includes: residual plot, Q-Q plot (normality check), histogram

# Q-Q (Quantile-Quantile) plot checks if residuals are normally distributed
# If residuals are normal, points should lie on diagonal line
# Deviations from diagonal = non-normal residuals

plot_residuals(
    y_test.values,  # True target values
    y_pred_test,  # Model predictions
    title="Linear Regression Residual Analysis"  # Plot title
)
# Creates multiple plots:
# 1. Residual vs Predicted (check for patterns)
# 2. Q-Q plot (check normality)
# 3. Histogram (distribution of residuals)

# Interpretation:
# - Normal residuals = good (assumption met)
# - Non-normal residuals = may need transformation or different model


## Real-World Application

Let's apply linear regression to a more complex scenario with hyperparameter tuning.

**Note on Hyperparameters:**
Basic LinearRegression has no hyperparameters to tune. However, we can:
- Use **Ridge Regression** (L2 regularization) to prevent overfitting
- Use **Lasso Regression** (L1 regularization) for feature selection
- Use **Elastic Net** (combines L1 and L2)

**Synthetic Data:**
We'll create synthetic data with known relationships to verify our model works correctly.


In [ ]:
# ============================================
# SYNTHETIC DATA: Testing on Known Relationships
# ============================================

# make_regression() creates synthetic data with known linear relationship
# Useful for:
# - Testing if model works correctly
# - Understanding model behavior
# - Demonstrating concepts

# Parameters:
# - n_samples=200: Create 200 data points
# - n_features=5: Each point has 5 features
# - noise=20: Add random noise (simulates real-world measurement error)
# - random_state=42: Reproducible random numbers
X_synthetic, y_synthetic = make_regression(
    n_samples=200, n_features=5, noise=20, random_state=42
)
# Returns: X (features) and y (target) with known linear relationship

# ============================================
# TRAIN/TEST SPLIT: Synthetic Data
# ============================================

# Split synthetic data into training (80%) and test (20%) sets
X_syn_train, X_syn_test, y_syn_train, y_syn_test = train_test_split(
    X_synthetic,  # Features
    y_synthetic,  # Target
    test_size=0.2,  # 20% for testing
    random_state=42  # Reproducible split
)

# ============================================
# TRAIN MODEL: On Synthetic Data
# ============================================

# Create and train linear regression on synthetic data
model_syn = LinearRegression()  # Create model
model_syn.fit(X_syn_train, y_syn_train)  # Train on synthetic training data

# ============================================
# EVALUATE MODEL: Synthetic Data Performance
# ============================================

# Make predictions on synthetic test data
y_syn_pred = model_syn.predict(X_syn_test)

# Calculate performance metrics
syn_rmse = np.sqrt(mean_squared_error(y_syn_test, y_syn_pred))  # RMSE
syn_r2 = r2_score(y_syn_test, y_syn_pred)  # R² score

print("Synthetic Dataset Results:")
print(f"  RMSE: {syn_rmse:.3f}")  # Prediction error
print(f"  R²: {syn_r2:.3f}")  # Goodness of fit

# ============================================
# VISUALIZE: Predicted vs Actual (Synthetic)
# ============================================

# Create scatter plot showing predictions vs actual values
plt.figure(figsize=(8, 6))  # Figure size: 8×6 inches

# Scatter plot: each point is (actual, predicted)
plt.scatter(y_syn_test, y_syn_pred, alpha=0.6)

# Perfect prediction line: y = x
plt.plot([y_syn_test.min(), y_syn_test.max()], 
         [y_syn_test.min(), y_syn_test.max()], 'r--', lw=2)

# Label axes and add title
plt.xlabel('Actual Values')  # X-axis: true values
plt.ylabel('Predicted Values')  # Y-axis: predictions
plt.title('Linear Regression on Synthetic Data')  # Chart title
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display plot

# Interpretation:
# - Points close to red line = model learned the relationship correctly
# - High R² = model explains most variance (good for synthetic data)


## Summary & Key Takeaways

### Key Concepts Learned

1. **Linear Regression Basics**
   - Models linear relationships between features and continuous target
   - Uses Ordinary Least Squares (OLS) to minimize sum of squared residuals
   - Provides interpretable coefficients

2. **Model Evaluation**
   - RMSE and MSE for regression error
   - R² score for goodness of fit
   - Cross-validation for robust performance estimation

3. **Assumptions & Diagnostics**
   - Check linearity, homoscedasticity, normality of residuals
   - Residual plots reveal model issues
   - Statistical tests validate assumptions

4. **Best Practices**
   - Scale features for meaningful coefficients
   - Use cross-validation to avoid overfitting
   - Check assumptions before trusting results
   - Compare with baseline models

### When to Use Linear Regression

✅ **Good for:**
- Continuous target variables
- Linear relationships
- Interpretability is important
- Small to medium datasets
- Baseline model for comparison

❌ **Not ideal for:**
- Non-linear relationships
- Very large datasets (use SGD variants)
- Highly correlated features (use regularization)
- Outliers (use robust regression)

### Next Steps

- Try **Ridge Regression** (L2 regularization) for multicollinearity
- Try **Lasso Regression** (L1 regularization) for feature selection
- Explore **Polynomial Regression** for non-linear relationships
- Consider **Elastic Net** for combining L1 and L2 regularization
